# Attention Scoring Functions
我们用的是“高斯核”来衡量相似度,但在高维空间里，计算欧式距离的平方不仅开销大，而且效果往往不如另一种方法。所以，我们需要设计新的评分函数 $a(\mathbf{q}, \mathbf{k})$

## Mask
由于文本序列的长度是不一样的，为了把它们拼成一个 Batch，我们必须在短句子的末尾填充无意义的 pad

如果把 pad 也放进 Softmax 里去算注意力，它也会分走一部分概率权重，这意味着模型把注意力浪费在了无意义的空白上

在做 Softmax 之前，我们强行把那些 pad 对应的注意力分数改成一个极小的负数。这样一过 Softmax，$e^{-1e9}$ 就会变成绝对的 0

## 加性注意力
如果你的 $\mathbf{q}$ 和 $\mathbf{k}$ 维度不一样，你是不可能直接对它们做内积的，这时候就需要用加性注意力
$$a(\mathbf{q}, \mathbf{k}) = \mathbf{w}_v^\top \text{tanh}(\mathbf{W}_q\mathbf{q} + \mathbf{W}_k\mathbf{k})$$
用矩阵 $\mathbf{W}_q$ 把 $\mathbf{q}$ 投影到维度 $h$

用矩阵 $\mathbf{W}_k$ 把 $\mathbf{k}$ 投影到维度 $h$

相加，过一个激活函数 tanh

最后乘上一个权重向量 $\mathbf{w}_v^\top$，把结果压扁成一个标量

## 缩放点积注意力
如果 Query 和 Key 维度相同（假设都是 $d$），就有了最简单、最暴力、GPU 最喜欢的相似度计算方法：点积
$$a(\mathbf{q}, \mathbf{k}) = \frac{\mathbf{q}^\top \mathbf{k}}{\sqrt{d}}$$
$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d}}\right)\mathbf{V}$$
为什么除以 $\sqrt{d}$ 

假设 $q$ 和 $k$ 里面的元素都是均值为 0、方差为 1 的独立随机变量。它们的点积结果 $q \cdot k = \sum_{i=1}^d q_i k_i$ 的方差会变成 $d$

如果向量维度 $d$ 很大，点积的结果就会极大或极小，放入 Softmax $\frac{e^x}{\sum e^y}$ 后，最大的那个值会把概率全部挤占，其他的都变成 0，这会导致梯度消失，模型彻底无法训练

把点积结果除以 $\sqrt{d}$，硬生生把方差拉回到 1，保证 Softmax 的梯度平稳健康

In [6]:
import torch
from torch import nn
import math

def sequence_mask(X, valid_len, value=0):
    """在序列中掩蔽不相关的项 (底层张量操作)"""
    maxlen = X.size(1)
    mask = torch.arange((maxlen), dtype=torch.float32,
                        device=X.device)[None, :] < valid_len[:, None]
    X[~mask] = value
    return X

def masked_softmax(X, valid_lens):
    """执行带有 Mask 的 Softmax 操作"""
    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            valid_lens = torch.repeat_interleave(valid_lens, shape[1])
        else:
            valid_lens = valid_lens.reshape(-1)
        # 这里的 -1e6 就是我们在上一节提到的，用来“毒死” pad 标签的极大负数！
        X = sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6)
        return nn.functional.softmax(X.reshape(shape), dim=-1)

# 第一步：定义注意力类 
class DotProductAttention(nn.Module):
    """缩放点积注意力"""
    def __init__(self, dropout, **kwargs):
        super(DotProductAttention, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        scores = torch.bmm(queries, keys.transpose(1,2)) / math.sqrt(d)
        # 这里调用了上面补全的 masked_softmax
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)

# 第二步：准备张量数据
torch.manual_seed(42)  
queries = torch.normal(0, 1, (2, 1, 2)) 
keys = torch.ones((2, 10, 2))
values = torch.ones((2, 10, 4))
valid_lens = torch.tensor([2, 6])

# 第三步：实例化并运行计算图
attention = DotProductAttention(dropout=0.5)
attention.eval()  
output = attention(queries, keys, values, valid_lens)

print("注意力权重张量形状:", attention.attention_weights.shape)
print("最终输出张量形状:", output.shape)
print("\n最终输出结果:\n", output)

注意力权重张量形状: torch.Size([2, 1, 10])
最终输出张量形状: torch.Size([2, 1, 4])

最终输出结果:
 tensor([[[1., 1., 1., 1.]],

        [[1., 1., 1., 1.]]])
